# 10x Feature Distribution Redundancy Pre Audit

This notebook performs distribution EDA and redundancy / group-proxy pre-audit only. It does not perform modeling, target prediction, train/test split, SHAP, Optuna, segmentation, feature removal, or feature selection decisions.

In [1]:
from pathlib import Path
import hashlib
import json
import math
import os
import shutil
import zipfile
from datetime import datetime

import numpy as np
import pandas as pd

STEP = '10x_feature_distribution_redundancy_pre_audit_260516'
ROOT = Path(r'C:/Code/ott-churn-prediction').resolve()
PARK = ROOT / 'park.ingyeom'
NOTEBOOK_PATH = PARK / 'notebook' / STEP / f'{STEP}.ipynb'
OUT = PARK / 'reports' / 'audits' / STEP
ZIP_PATH = PARK / 'zip' / f'{STEP}_review_package.zip'
DATA_DIR = PARK / 'data'
FIG_DIR = OUT / 'figures'

F06 = PARK / 'reports' / 'audits' / '06x_dataset_generation_260515'
F07 = PARK / 'reports' / 'audits' / '07x_feature_mapping_AARRR_260515'
F08 = PARK / 'reports' / 'audits' / '08x_promotion_nonpromotion_EDA_260516'
F09 = PARK / 'reports' / 'audits' / '09x_promotion_repurchase_2x2_EDA_260516'

OUT.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
(PARK / 'zip').mkdir(parents=True, exist_ok=True)

started_at = datetime.now().isoformat(timespec='seconds')
warnings_list = []
errors_list = []
print('10x start:', started_at)
print('Notebook:', NOTEBOOK_PATH)
print('Output:', OUT)

10x start: 2026-05-16T02:36:37
Notebook: C:\Code\ott-churn-prediction\park.ingyeom\notebook\10x_feature_distribution_redundancy_pre_audit_260516\10x_feature_distribution_redundancy_pre_audit_260516.ipynb
Output: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\10x_feature_distribution_redundancy_pre_audit_260516


In [2]:
def read_csv(path, **kwargs):
    return pd.read_csv(path, encoding='utf-8-sig', **kwargs)

def write_csv(df, name):
    path = OUT / name
    df.to_csv(path, index=False, encoding='utf-8-sig')
    return path

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def file_fingerprint(path):
    if not path.exists():
        return {'sha256': None, 'mtime': None, 'size': None, 'exists': False, 'error': None}
    try:
        stat = path.stat()
        return {
            'sha256': sha256_file(path),
            'mtime': datetime.fromtimestamp(stat.st_mtime).isoformat(timespec='seconds'),
            'size': stat.st_size,
            'exists': True,
            'error': None,
        }
    except Exception as exc:
        return {'sha256': None, 'mtime': None, 'size': None, 'exists': path.exists(), 'error': str(exc)}

def final_checks_pass(path):
    if not path.exists():
        return False
    df = read_csv(path)
    status_col = 'status' if 'status' in df.columns else None
    if status_col is None:
        return False
    return (df[status_col].astype(str).str.upper() == 'PASS').all()

def safe_bool(v):
    return str(v).strip().lower() in {'true', '1', 'yes', 'y'}

def to_num(s):
    return pd.to_numeric(s, errors='coerce')

def is_binary_series(s):
    vals = sorted(pd.Series(s).dropna().unique().tolist())
    return len(vals) > 0 and set(vals).issubset({0, 1, 0.0, 1.0, False, True})

def infer_type(df, col):
    if col not in df.columns:
        return 'missing'
    if col in {'USER_KEY'}:
        return 'group_key'
    if col == 'is_repurchase':
        return 'target'
    if col == 'is_promotion':
        return 'split_key'
    num = to_num(df[col])
    if num.notna().sum() == 0:
        return 'non_numeric_or_empty'
    if is_binary_series(num):
        return 'binary'
    return 'numeric'

def q(series, p):
    s = to_num(series).dropna()
    if s.empty:
        return np.nan
    return float(s.quantile(p))

def iqr_outlier_rate(series):
    s = to_num(series).dropna()
    if s.empty:
        return (0, np.nan)
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    if pd.isna(iqr) or iqr == 0:
        return (0, 0.0)
    mask = (s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)
    return (int(mask.sum()), float(mask.mean()))

def joined(items):
    vals = [str(x) for x in items if str(x) not in {'', 'nan', 'None'}]
    return '; '.join(vals)

raw_files = [
    ('derived_source_master', DATA_DIR / '(광일)Membership_v2_with_derived_features.csv'),
    ('raw_membership', DATA_DIR / 'Membership_v2.csv'),
    ('raw_view_history', DATA_DIR / 'View_History_v2.csv'),
    ('raw_user_mapping', DATA_DIR / 'User_Mapping_v2.csv'),
    ('raw_movie_master', DATA_DIR / 'Movie_Master_v2.csv'),
    ('raw_membership_train', DATA_DIR / 'Membership_train.csv'),
    ('raw_feature_union_dictionary', DATA_DIR / '변수_합집합_비교_v3.csv'),
]
fp_before = {str(path): file_fingerprint(path) for _, path in raw_files}
print('source fingerprint before captured:', len(fp_before))

source fingerprint before captured: 7


In [3]:
required = {
    F06: ['06x_conservative_dataset.csv','06x_expanded_dataset.csv','06x_model_feature_lists.csv','06x_dataset_schema_conservative.csv','06x_dataset_schema_expanded.csv','06x_scope_feature_policy.csv','06x_caveat_register.csv','06x_final_checks.csv'],
    F07: ['07x_feature_mapping_master.csv','07x_AARRR_summary_by_feature_set.csv','07x_conservative_AARRR_mapping.csv','07x_expanded_AARRR_mapping.csv','07x_scope_policy_handoff.csv','07x_caveat_handoff.csv','07x_downstream_EDA_handoff.csv','07x_final_checks.csv'],
    F08: ['08x_dataset_scope_summary.csv','08x_promotion_target_summary.csv','08x_numeric_feature_group_comparison.csv','08x_binary_feature_group_comparison.csv','08x_feature_family_summary.csv','08x_AARRR_stage_summary.csv','08x_top_observed_differences_for_review.csv','08x_caveat_and_claim_guardrail.csv','08x_downstream_handoff.csv','08x_redundancy_audit_handoff.csv','08x_final_checks.csv'],
    F09: ['09x_2x2_cohort_summary.csv','09x_2x2_target_rate_summary.csv','09x_numeric_2x2_feature_profile.csv','09x_binary_2x2_feature_profile.csv','09x_within_promotion_repurchase_comparison.csv','09x_within_nonpromotion_repurchase_comparison.csv','09x_between_promotion_status_within_repurchase_comparison.csv','09x_feature_family_2x2_summary.csv','09x_AARRR_stage_2x2_summary.csv','09x_top_2x2_observed_differences_for_review.csv','09x_context_profile_proxy_risk_review.csv','09x_usage_retention_2x2_review.csv','09x_caveat_and_claim_guardrail.csv','09x_downstream_handoff.csv','09x_redundancy_audit_handoff.csv','09x_final_checks.csv'],
}

preflight = []
for folder, files in required.items():
    preflight.append({'check': f'{folder.name} folder exists', 'status': 'PASS' if folder.exists() else 'FAIL', 'detail': str(folder)})
    missing = [f for f in files if not (folder / f).exists()]
    preflight.append({'check': f'required {folder.name} files exist', 'status': 'PASS' if not missing else 'FAIL', 'detail': joined(missing)})

conservative = read_csv(F06 / '06x_conservative_dataset.csv')
expanded = read_csv(F06 / '06x_expanded_dataset.csv')
feature_lists = read_csv(F06 / '06x_model_feature_lists.csv')
mapping = read_csv(F07 / '07x_feature_mapping_master.csv')
top08 = read_csv(F08 / '08x_top_observed_differences_for_review.csv')
top09 = read_csv(F09 / '09x_top_2x2_observed_differences_for_review.csv')
proxy09 = read_csv(F09 / '09x_context_profile_proxy_risk_review.csv')

alignment = conservative['USER_KEY'].astype(str).equals(expanded['USER_KEY'].astype(str)) and conservative['is_repurchase'].equals(expanded['is_repurchase'])
preflight.extend([
    {'check': '06x final checks pass', 'status': 'PASS' if final_checks_pass(F06 / '06x_final_checks.csv') else 'FAIL', 'detail': ''},
    {'check': '07x final checks pass', 'status': 'PASS' if final_checks_pass(F07 / '07x_final_checks.csv') else 'FAIL', 'detail': ''},
    {'check': '08x final checks pass', 'status': 'PASS' if final_checks_pass(F08 / '08x_final_checks.csv') else 'FAIL', 'detail': ''},
    {'check': '09x final checks pass', 'status': 'PASS' if final_checks_pass(F09 / '09x_final_checks.csv') else 'FAIL', 'detail': ''},
    {'check': 'conservative dataset loaded', 'status': 'PASS', 'detail': str(conservative.shape)},
    {'check': 'expanded dataset loaded', 'status': 'PASS', 'detail': str(expanded.shape)},
    {'check': 'conservative_expanded_row_alignment_verified', 'status': 'PASS' if alignment else 'FAIL', 'detail': 'USER_KEY and target aligned'},
    {'check': 'is_promotion split available', 'status': 'PASS' if 'is_promotion' in expanded.columns else 'FAIL', 'detail': 'expanded dataset'},
    {'check': 'is_repurchase target available', 'status': 'PASS' if 'is_repurchase' in expanded.columns and 'is_repurchase' in conservative.columns else 'FAIL', 'detail': ''},
    {'check': '2x2 cohort labels available for analysis only', 'status': 'PASS' if {'is_promotion','is_repurchase'}.issubset(expanded.columns) else 'FAIL', 'detail': 'computed as masks only, not saved as new dataset features'},
    {'check': 'source fingerprint before captured', 'status': 'PASS' if len(fp_before) == len(raw_files) else 'FAIL', 'detail': str(len(fp_before))},
    {'check': 'output folder created', 'status': 'PASS' if OUT.exists() else 'FAIL', 'detail': str(OUT)},
    {'check': 'stop_reason', 'status': 'PASS', 'detail': 'none'},
])
write_csv(pd.DataFrame(preflight), '10x_preflight_input_validation.csv')
print('loaded conservative:', conservative.shape, 'expanded:', expanded.shape)
print('row alignment:', alignment)

loaded conservative: (23079, 24) expanded: (23079, 82)
row alignment: True


In [4]:
meta_cols = ['feature_set_name','safe_model_feature_name','original_feature_name','role','use_as_feature','AARRR_stage','feature_family','timing_family','caveat_flag','caveat_reason','needs_user_review']
meta = mapping.copy()
for col in meta_cols:
    if col not in meta.columns:
        meta[col] = ''
meta = meta[meta_cols].drop_duplicates(['feature_set_name','safe_model_feature_name'], keep='first')

def feature_rows(feature_set_name):
    rows = feature_lists[feature_lists['feature_set_name'].eq(feature_set_name)].copy()
    if rows.empty:
        rows = meta[meta['feature_set_name'].eq(feature_set_name)].copy()
    return rows

sets = {
    'conservative_safe_22': conservative,
    'expanded_feature_set': expanded,
}

catalog_rows = []
feature_info = {}
for fs, df in sets.items():
    rows = feature_rows(fs)
    for _, r in rows.iterrows():
        col = r.get('safe_model_feature_name', r.get('original_feature_name', ''))
        if col not in df.columns:
            continue
        m = meta[(meta['feature_set_name'].eq(fs)) & (meta['safe_model_feature_name'].eq(col))]
        md = m.iloc[0].to_dict() if not m.empty else {}
        ser = df[col]
        typ = infer_type(df, col)
        num = to_num(ser)
        row = {
            'feature_set_name': fs,
            'safe_model_feature_name': col,
            'original_feature_name': r.get('original_feature_name', md.get('original_feature_name', col)),
            'role': r.get('role', md.get('role', 'feature')),
            'use_as_feature': r.get('use_as_feature', md.get('use_as_feature', '')),
            'AARRR_stage': md.get('AARRR_stage', ''),
            'feature_family': md.get('feature_family', ''),
            'timing_family': md.get('timing_family', ''),
            'inferred_feature_type': typ,
            'unique_count': int(ser.nunique(dropna=True)),
            'missing_count': int(ser.isna().sum()),
            'missing_rate': float(ser.isna().mean()),
            'zero_count': int((num == 0).sum()) if num.notna().any() else np.nan,
            'zero_rate': float((num == 0).mean()) if num.notna().any() else np.nan,
            'min': float(num.min()) if num.notna().any() else np.nan,
            'max': float(num.max()) if num.notna().any() else np.nan,
            'notes': 'group key only' if col == 'USER_KEY' else ('target only' if col == 'is_repurchase' else ('split key / scope conditional feature only' if col == 'is_promotion' else '')),
        }
        catalog_rows.append(row)
        feature_info[(fs, col)] = row

catalog = pd.DataFrame(catalog_rows)
write_csv(catalog, '10x_feature_distribution_catalog.csv')
print('catalog rows:', len(catalog))

catalog rows: 106


In [5]:
def use_as_feature_value(v):
    return str(v).strip().lower() in {'yes', 'true', '1'}

def feature_cols(fs, kind=None, include_binary_in_numeric=False):
    sub = catalog[(catalog['feature_set_name'].eq(fs)) & (catalog['role'].eq('feature')) & (catalog['use_as_feature'].map(use_as_feature_value))].copy()
    if kind == 'binary':
        sub = sub[sub['inferred_feature_type'].eq('binary')]
    elif kind == 'numeric':
        if include_binary_in_numeric:
            sub = sub[sub['inferred_feature_type'].isin(['numeric','binary'])]
        else:
            sub = sub[sub['inferred_feature_type'].eq('numeric')]
    return sub['safe_model_feature_name'].tolist()

def info(fs, col, key):
    return feature_info.get((fs, col), {}).get(key, '')

def caveat_for_feature(fs, col):
    reasons = []
    fam = info(fs, col, 'feature_family')
    if 'genre' in str(fam).lower() or 'ratio' in col and any(x in col for x in ['action','family','drama','thriller','fantasy','comedy','romance','horror','documentary','historical','other']):
        reasons.append('Movie_Master same MOVIE_NUM multi-category caveat applies to content/genre interpretation')
    if col == 'old_movie_ratio_5y':
        reasons.append('old_movie_ratio_5y 9-row mismatch caveat applies')
    if 'under_1m' in col or 'under_5m' in col:
        reasons.append('under_1m/under_5m are <= threshold ratio features, not business effect evidence')
    if 'cold_start' in col:
        reasons.append('cold_start_fixed row-level first-watch timing caveat applies')
    return joined(reasons)

def numeric_summary(fs, df, col):
    s = to_num(df[col]).dropna()
    out_count, out_rate = iqr_outlier_rate(df[col])
    q99 = q(df[col], 0.99)
    maxv = float(s.max()) if not s.empty else np.nan
    zero_rate = float((to_num(df[col]) == 0).mean()) if to_num(df[col]).notna().any() else np.nan
    caveat = caveat_for_feature(fs, col)
    return {
        'feature_set_name': fs, 'safe_model_feature_name': col,
        'AARRR_stage': info(fs, col, 'AARRR_stage'), 'feature_family': info(fs, col, 'feature_family'), 'timing_family': info(fs, col, 'timing_family'),
        'n': int(s.shape[0]), 'mean': float(s.mean()) if not s.empty else np.nan, 'std': float(s.std()) if len(s) > 1 else np.nan,
        'min': float(s.min()) if not s.empty else np.nan, 'q01': q(df[col], .01), 'q05': q(df[col], .05), 'q10': q(df[col], .10), 'q25': q(df[col], .25), 'median': q(df[col], .50), 'q75': q(df[col], .75), 'q90': q(df[col], .90), 'q95': q(df[col], .95), 'q99': q99, 'max': maxv,
        'iqr': q(df[col], .75) - q(df[col], .25) if pd.notna(q(df[col], .75)) and pd.notna(q(df[col], .25)) else np.nan,
        'skewness': float(s.skew()) if len(s) > 2 else np.nan,
        'zero_count': int((to_num(df[col]) == 0).sum()), 'zero_rate': zero_rate,
        'nonzero_count': int((to_num(df[col]) != 0).sum()), 'nonzero_rate': float((to_num(df[col]) != 0).mean()),
        'outlier_iqr_count': out_count, 'outlier_iqr_rate': out_rate,
        'caveat_flag': bool(caveat), 'caveat_reason': caveat,
        'needs_user_review': bool(caveat), 'notes': 'distribution EDA only; not feature importance evidence',
    }

def binary_summary(fs, df, col):
    s = to_num(df[col])
    n = int(s.notna().sum())
    pos = int((s == 1).sum())
    neg = int((s == 0).sum())
    rate = pos / n if n else np.nan
    caveat = caveat_for_feature(fs, col)
    near = bool(pd.notna(rate) and (rate <= 0.01 or rate >= 0.99))
    return {
        'feature_set_name': fs, 'safe_model_feature_name': col,
        'AARRR_stage': info(fs, col, 'AARRR_stage'), 'feature_family': info(fs, col, 'feature_family'), 'timing_family': info(fs, col, 'timing_family'),
        'n': n, 'positive_count': pos, 'positive_rate': rate, 'negative_count': neg, 'negative_rate': neg / n if n else np.nan,
        'missing_count': int(s.isna().sum()), 'missing_rate': float(s.isna().mean()),
        'rare_positive_flag': bool(pd.notna(rate) and rate <= 0.01), 'near_constant_flag': near,
        'caveat_flag': bool(caveat), 'caveat_reason': caveat,
        'needs_user_review': bool(caveat or near), 'notes': 'near_constant threshold: positive_rate <= 0.01 or >= 0.99',
    }

num_overall, bin_overall = [], []
for fs, df in sets.items():
    for col in feature_cols(fs, 'numeric'):
        num_overall.append(numeric_summary(fs, df, col))
    for col in feature_cols(fs, 'binary'):
        bin_overall.append(binary_summary(fs, df, col))

num_overall_df = pd.DataFrame(num_overall)
bin_overall_df = pd.DataFrame(bin_overall)
write_csv(num_overall_df, '10x_numeric_distribution_overall.csv')
write_csv(bin_overall_df, '10x_binary_distribution_overall.csv')
print('numeric overall rows:', len(num_overall_df), 'binary overall rows:', len(bin_overall_df))

numeric overall rows: 69 binary overall rows: 32


In [6]:
group_masks = {
    'overall': expanded.index == expanded.index,
    'promotion_only': to_num(expanded['is_promotion']).eq(1),
    'nonpromotion_only': to_num(expanded['is_promotion']).eq(0),
    'promotion_repurchase': to_num(expanded['is_promotion']).eq(1) & to_num(expanded['is_repurchase']).eq(1),
    'promotion_nonrepurchase': to_num(expanded['is_promotion']).eq(1) & to_num(expanded['is_repurchase']).eq(0),
    'nonpromotion_repurchase': to_num(expanded['is_promotion']).eq(0) & to_num(expanded['is_repurchase']).eq(1),
    'nonpromotion_nonrepurchase': to_num(expanded['is_promotion']).eq(0) & to_num(expanded['is_repurchase']).eq(0),
}

def group_df(fs, df, mask):
    mask_values = np.asarray(mask)
    return df.loc[mask_values].copy()

num_group_rows, bin_group_rows = [], []
for fs, df in sets.items():
    for gname, mask in group_masks.items():
        gdf = group_df(fs, df, mask)
        for col in feature_cols(fs, 'numeric'):
            r = numeric_summary(fs, gdf, col)
            num_group_rows.append({k: r[k] for k in ['feature_set_name','safe_model_feature_name','AARRR_stage','feature_family','timing_family','n','mean','std','min','q05','q25','median','q75','q95','max','zero_rate','outlier_iqr_rate','caveat_flag','caveat_reason','needs_user_review','notes']} | {'group_name': gname})
        for col in feature_cols(fs, 'binary'):
            r = binary_summary(fs, gdf, col)
            bin_group_rows.append({k: r[k] for k in ['feature_set_name','safe_model_feature_name','AARRR_stage','feature_family','timing_family','n','positive_count','positive_rate','negative_count','negative_rate','rare_positive_flag','near_constant_flag','caveat_flag','caveat_reason','needs_user_review','notes']} | {'group_name': gname})

num_group_df = pd.DataFrame(num_group_rows)
bin_group_df = pd.DataFrame(bin_group_rows)
num_group_df = num_group_df[['feature_set_name','group_name','safe_model_feature_name','AARRR_stage','feature_family','timing_family','n','mean','std','min','q05','q25','median','q75','q95','max','zero_rate','outlier_iqr_rate','caveat_flag','caveat_reason','needs_user_review','notes']]
bin_group_df = bin_group_df[['feature_set_name','group_name','safe_model_feature_name','AARRR_stage','feature_family','timing_family','n','positive_count','positive_rate','negative_count','negative_rate','rare_positive_flag','near_constant_flag','caveat_flag','caveat_reason','needs_user_review','notes']]
write_csv(num_group_df, '10x_numeric_distribution_by_group.csv')
write_csv(bin_group_df, '10x_binary_distribution_by_group.csv')
print('group distribution rows:', len(num_group_df), len(bin_group_df))

group distribution rows: 483 224


In [7]:
zero_tail_rows = []
for _, r in num_overall_df.iterrows():
    q99 = r['q99']
    maxv = r['max']
    ratio = maxv / q99 if pd.notna(q99) and q99 not in [0, 0.0] else np.nan
    tail_flag = bool((pd.notna(r['outlier_iqr_rate']) and r['outlier_iqr_rate'] >= 0.05) or (pd.notna(ratio) and ratio >= 3))
    zero_flag = bool(pd.notna(r['zero_rate']) and r['zero_rate'] >= 0.50)
    zero_tail_rows.append({
        'feature_set_name': r['feature_set_name'], 'safe_model_feature_name': r['safe_model_feature_name'], 'AARRR_stage': r['AARRR_stage'], 'feature_family': r['feature_family'], 'timing_family': r['timing_family'],
        'zero_rate': r['zero_rate'], 'nonzero_rate': r['nonzero_rate'], 'q95': r['q95'], 'q99': q99, 'max': maxv, 'max_to_q99_ratio': ratio, 'outlier_iqr_rate': r['outlier_iqr_rate'],
        'tail_risk_flag': tail_flag, 'zero_inflation_flag': zero_flag,
        'recommended_modeling_caution': joined(['check transformations or robust model sensitivity' if tail_flag else '', 'zero-heavy distribution: review scope-specific handling' if zero_flag else '']),
        'removal_allowed': False, 'user_approval_required': True,
        'notes': 'pre-audit diagnostic only; no removal decision',
    })
zero_tail_df = pd.DataFrame(zero_tail_rows)
write_csv(zero_tail_df, '10x_zero_inflation_and_tail_risk_audit.csv')

proxy_features = ['is_user_verified','payment_is_ios','age_group','payment_is_mobile','is_female','is_premium','is_male','payment_is_pc']
proxy_rows = []
for fs, df in sets.items():
    for col in sorted(set(proxy_features + feature_cols(fs, 'binary'))):
        if col not in df.columns:
            continue
        vals = {}
        for gname, mask in group_masks.items():
            gdf = group_df(fs, df, mask)
            s = to_num(gdf[col])
            vals[gname] = float(s.mean()) if s.notna().any() else np.nan
        all_rates = [v for v in vals.values() if pd.notna(v)]
        near_overall = bool(pd.notna(vals['overall']) and (vals['overall'] <= 0.01 or vals['overall'] >= 0.99))
        near_any = any((v <= 0.01 or v >= 0.99) for v in all_rates)
        spread = max(all_rates) - min(all_rates) if all_rates else np.nan
        group_proxy = bool(col in proxy_features or (pd.notna(spread) and spread >= 0.10 and any(x in col for x in ['payment','age','female','male','verified','premium'])))
        proxy_rows.append({
            'feature_set_name': fs, 'safe_model_feature_name': col, 'AARRR_stage': info(fs, col, 'AARRR_stage'), 'feature_family': info(fs, col, 'feature_family'), 'timing_family': info(fs, col, 'timing_family'),
            'overall_rate_or_mean': vals['overall'], 'promotion_rate_or_mean': vals['promotion_only'], 'nonpromotion_rate_or_mean': vals['nonpromotion_only'],
            'promotion_repurchase_rate_or_mean': vals['promotion_repurchase'], 'promotion_nonrepurchase_rate_or_mean': vals['promotion_nonrepurchase'],
            'nonpromotion_repurchase_rate_or_mean': vals['nonpromotion_repurchase'], 'nonpromotion_nonrepurchase_rate_or_mean': vals['nonpromotion_nonrepurchase'],
            'near_constant_overall': near_overall, 'near_constant_any_group': near_any,
            'group_proxy_risk': group_proxy, 'leakage_suspect_pre_audit': bool(col == 'is_churn_prevented'),
            'redundancy_review_needed': bool(near_any), 'user_approval_required': True,
            'recommended_next_step': 'carry to 11x scope sensitivity and actual model input review; no automatic exclusion',
            'notes': 'group proxy risk is not a removal decision',
        })
proxy_df = pd.DataFrame(proxy_rows)
write_csv(proxy_df, '10x_near_constant_and_group_proxy_audit.csv')
print('zero/tail rows:', len(zero_tail_df), 'proxy rows:', len(proxy_df))

zero/tail rows: 69 proxy rows: 33


In [8]:
def corr_input(fs, df):
    cols = [c for c in feature_cols(fs, 'numeric', include_binary_in_numeric=True) if c not in {'USER_KEY','is_repurchase','is_promotion'}]
    keep = []
    for c in cols:
        s = to_num(df[c])
        if s.notna().sum() >= 3 and s.nunique(dropna=True) > 1:
            keep.append(c)
    return df[keep].apply(to_num)

corr_rows = []
for fs, df in sets.items():
    mat = corr_input(fs, df)
    pear = mat.corr(method='pearson')
    spear = mat.corr(method='spearman')
    cols = list(mat.columns)
    for i, a in enumerate(cols):
        for b in cols[i+1:]:
            pc, sc = pear.loc[a, b], spear.loc[a, b]
            high = bool((pd.notna(pc) and abs(pc) >= 0.85) or (pd.notna(sc) and abs(sc) >= 0.85))
            same = info(fs, a, 'feature_family') == info(fs, b, 'feature_family')
            corr_rows.append({
                'feature_set_name': fs, 'feature_a': a, 'feature_b': b,
                'AARRR_stage_a': info(fs, a, 'AARRR_stage'), 'AARRR_stage_b': info(fs, b, 'AARRR_stage'),
                'feature_family_a': info(fs, a, 'feature_family'), 'feature_family_b': info(fs, b, 'feature_family'),
                'pearson_corr': pc, 'spearman_corr': sc, 'abs_pearson_corr': abs(pc) if pd.notna(pc) else np.nan, 'abs_spearman_corr': abs(sc) if pd.notna(sc) else np.nan,
                'high_corr_flag': high, 'same_family_flag': same, 'redundancy_review_needed': high,
                'removal_allowed': False, 'user_approval_required': True,
                'notes': 'correlation diagnostic only; high correlation does not imply automatic removal',
            })
corr_df = pd.DataFrame(corr_rows)
write_csv(corr_df, '10x_pairwise_correlation_audit.csv')

def clusters_from_edges(features, edges):
    parent = {f: f for f in features}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra
    for a, b in edges:
        union(a, b)
    groups = {}
    for f in features:
        groups.setdefault(find(f), []).append(f)
    return [sorted(v) for v in groups.values() if len(v) > 1]

cluster_rows = []
for fs in sets:
    sub = corr_df[corr_df['feature_set_name'].eq(fs)]
    edges = list(sub[sub['high_corr_flag']][['feature_a','feature_b']].itertuples(index=False, name=None))
    features = sorted(set(sub['feature_a']).union(set(sub['feature_b'])))
    for idx, cl in enumerate(clusters_from_edges(features, edges), start=1):
        fams = [info(fs, c, 'feature_family') for c in cl]
        stages = sorted(set(info(fs, c, 'AARRR_stage') for c in cl))
        pairs = sub[sub['feature_a'].isin(cl) & sub['feature_b'].isin(cl)]
        maxcorr = pairs[['abs_pearson_corr','abs_spearman_corr']].max(axis=1).max() if not pairs.empty else np.nan
        dominant = pd.Series(fams).mode().iloc[0] if fams else ''
        cluster_rows.append({
            'feature_set_name': fs, 'cluster_id': f'{fs}_corr_cluster_{idx:02d}', 'cluster_feature_list': joined(cl), 'cluster_size': len(cl),
            'dominant_feature_family': dominant, 'AARRR_stages': joined(stages), 'max_abs_corr_in_cluster': maxcorr,
            'reason': 'features connected by abs pearson or spearman correlation >= 0.85',
            'modeling_risk': 'coefficient instability or model-family-specific redundancy sensitivity possible',
            'SHAP_interpretation_risk': 'correlated feature importance may split across related features',
            'removal_allowed': False, 'user_approval_required': True,
            'recommended_next_step': 'review as family/cluster in 11x and SHAP; do not remove here', 'notes': 'cluster is not a feature selection decision',
        })
cluster_df = pd.DataFrame(cluster_rows)
write_csv(cluster_df, '10x_redundancy_cluster_pre_audit.csv')
print('correlation rows:', len(corr_df), 'clusters:', len(cluster_df))

correlation rows: 3312 clusters: 15


In [9]:
def vif_for_matrix(X):
    values = X.to_numpy(dtype=float)
    out = {}
    for j, col in enumerate(X.columns):
        y = values[:, j]
        others = np.delete(values, j, axis=1)
        design = np.column_stack([np.ones(len(y)), others])
        try:
            beta, *_ = np.linalg.lstsq(design, y, rcond=None)
            pred = design @ beta
            ss_res = float(np.sum((y - pred) ** 2))
            ss_tot = float(np.sum((y - y.mean()) ** 2))
            r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
            out[col] = np.inf if pd.notna(r2) and r2 >= 0.999999 else (1 / (1 - r2) if pd.notna(r2) else np.nan)
        except Exception as exc:
            warnings_list.append(f'VIF failed for {col}: {exc}')
            out[col] = np.nan
    return out

vif_rows = []
for fs, df in sets.items():
    candidates = [c for c in feature_cols(fs, 'numeric', include_binary_in_numeric=True) if c not in {'USER_KEY','is_repurchase','is_promotion'}]
    included, excluded = [], {}
    for c in candidates:
        s = to_num(df[c])
        rate = float(s.mean()) if is_binary_series(s) and s.notna().any() else np.nan
        if s.notna().sum() < 3:
            excluded[c] = 'too_few_non_missing_values'
        elif s.nunique(dropna=True) <= 1:
            excluded[c] = 'constant_feature'
        elif pd.notna(rate) and (rate <= 0.01 or rate >= 0.99):
            excluded[c] = 'near_constant_binary_feature'
        else:
            included.append(c)
    X = df[included].apply(to_num)
    X = X.fillna(X.median(numeric_only=True)).fillna(0)
    vifs = vif_for_matrix(X) if len(included) >= 2 else {}
    for c in candidates:
        excluded_flag = c in excluded or c not in vifs
        val = np.nan if excluded_flag else vifs.get(c, np.nan)
        if excluded_flag:
            bucket = 'not_computed'
        elif pd.isna(val):
            bucket = 'not_computed'
        elif np.isinf(val) or val >= 20:
            bucket = 'extreme'
        elif val >= 10:
            bucket = 'high'
        elif val >= 5:
            bucket = 'moderate'
        else:
            bucket = 'low'
        vif_rows.append({
            'feature_set_name': fs, 'safe_model_feature_name': c, 'AARRR_stage': info(fs, c, 'AARRR_stage'), 'feature_family': info(fs, c, 'feature_family'), 'timing_family': info(fs, c, 'timing_family'),
            'vif_value': val, 'vif_risk_bucket': bucket, 'excluded_from_vif': excluded_flag, 'excluded_from_vif_reason': excluded.get(c, '' if c in vifs else 'not_enough_included_features'),
            'redundancy_review_needed': bucket in {'high','extreme'} or excluded.get(c, '') == 'near_constant_binary_feature',
            'removal_allowed': False, 'user_approval_required': True,
            'notes': 'VIF diagnostic for LogisticRegression interpretability risk only; no automatic removal',
        })
vif_df = pd.DataFrame(vif_rows)
write_csv(vif_df, '10x_vif_pre_audit.csv')

dup_rows = []
for fs, df in sets.items():
    cols = [c for c in feature_cols(fs, 'numeric', include_binary_in_numeric=True) if c not in {'USER_KEY','is_repurchase','is_promotion'}]
    high_pairs = corr_df[(corr_df['feature_set_name'].eq(fs)) & ((corr_df['abs_pearson_corr'] >= 0.98) | (corr_df['abs_spearman_corr'] >= 0.98))]
    candidate_pairs = set(tuple(x) for x in high_pairs[['feature_a','feature_b']].itertuples(index=False, name=None))
    for i, a in enumerate(cols):
        for b in cols[i+1:]:
            shared = len(set(a.split('_')).intersection(set(b.split('_'))))
            if (a, b) not in candidate_pairs and shared < 2:
                continue
            sa, sb = to_num(df[a]), to_num(df[b])
            valid = sa.notna() & sb.notna()
            match = float((sa[valid].values == sb[valid].values).mean()) if valid.any() else np.nan
            csub = corr_df[(corr_df['feature_set_name'].eq(fs)) & (corr_df['feature_a'].eq(a)) & (corr_df['feature_b'].eq(b))]
            abs_corr = csub[['abs_pearson_corr','abs_spearman_corr']].max(axis=1).iloc[0] if not csub.empty else np.nan
            risk = bool((pd.notna(match) and match >= 0.99) or (pd.notna(abs_corr) and abs_corr >= 0.98))
            if risk or shared >= 3:
                dup_rows.append({
                    'feature_set_name': fs, 'feature_a': a, 'feature_b': b,
                    'reason': joined(['high value match' if pd.notna(match) and match >= 0.99 else '', 'very high correlation' if pd.notna(abs_corr) and abs_corr >= 0.98 else '', 'similar name tokens' if shared >= 3 else '']),
                    'exact_value_match_rate': match, 'abs_corr': abs_corr, 'same_family_flag': info(fs, a, 'feature_family') == info(fs, b, 'feature_family'),
                    'duplicate_like_risk': risk, 'removal_allowed': False, 'user_approval_required': True,
                    'recommended_next_step': 'review duplicate-like semantics in 11x; no removal in 10x', 'notes': 'candidate only',
                })
dup_df = pd.DataFrame(dup_rows)
write_csv(dup_df, '10x_duplicate_like_feature_audit.csv')
print('vif rows:', len(vif_df), 'duplicate-like rows:', len(dup_df))

vif rows: 101 duplicate-like rows: 76


In [10]:
def colset_from(df):
    vals = []
    for col in df.columns:
        if 'feature' in col.lower():
            vals += df[col].dropna().astype(str).tolist()
    return set(vals)
top_features = colset_from(top08).union(colset_from(top09)).union(colset_from(proxy09))
leak_rows = []
for fs, df in sets.items():
    for c in feature_cols(fs, None):
        name = c.lower()
        reasons = []
        if c == 'is_churn_prevented': reasons.append('historical context / target-proxy review required')
        if 'recency' in name: reasons.append('recency timing availability at day21 must be rechecked')
        if any(x in name for x in ['total_', 'max_', 'avg_', 'median_', 'std_', 'watch_days', 'active_ratio']): reasons.append('summary usage timing window must remain day0-20 only')
        if any(x in name for x in ['payment', 'age_group', 'female', 'male', 'verified', 'premium']): reasons.append('context/profile/payment proxy risk')
        if c in top_features: reasons.append('observed in 08x/09x difference review; not importance evidence')
        level = 'high' if c == 'is_churn_prevented' else ('medium' if reasons else 'low')
        leak_rows.append({
            'feature_set_name': fs, 'safe_model_feature_name': c, 'AARRR_stage': info(fs, c, 'AARRR_stage'), 'feature_family': info(fs, c, 'feature_family'), 'timing_family': info(fs, c, 'timing_family'),
            'reason_for_review': joined(reasons) or 'routine pre-audit coverage',
            'availability_at_day21_assumption': 'requires 11x preflight confirmation against source construction; 10x does not make final leakage decision',
            'leakage_suspect_level': level,
            'evidence_from_05y_06x_07x': '06x feature list and 07x mapping reviewed; source policy caveats carried forward',
            'evidence_from_08x_09x': '08x/09x observed-difference and proxy-risk handoffs reviewed where available',
            'allowed_for_modeling_currently': 'pending 11x preflight and user approval; not decided by 10x',
            'user_approval_required': True,
            'recommended_next_step': 'verify actual 11x model input feature list and timing availability before modeling',
            'notes': 'pre-audit only, not final leakage ruling',
        })
leak_df = pd.DataFrame(leak_rows)
write_csv(leak_df, '10x_target_leakage_suspect_pre_audit.csv')

family_rows = []
for (fs, stage, fam), subcat in catalog[catalog['role'].eq('feature')].groupby(['feature_set_name','AARRR_stage','feature_family'], dropna=False):
    feats = subcat['safe_model_feature_name'].tolist()
    zeros = catalog[(catalog['feature_set_name'].eq(fs)) & (catalog['safe_model_feature_name'].isin(feats))]['zero_rate']
    family_rows.append({
        'feature_set_name': fs, 'AARRR_stage': stage, 'feature_family': fam, 'feature_count': len(feats),
        'numeric_count': int(subcat['inferred_feature_type'].eq('numeric').sum()), 'binary_count': int(subcat['inferred_feature_type'].eq('binary').sum()),
        'avg_zero_rate': float(zeros.mean()) if len(zeros) else np.nan, 'max_zero_rate': float(zeros.max()) if len(zeros) else np.nan,
        'near_constant_count': int(proxy_df[(proxy_df['feature_set_name'].eq(fs)) & (proxy_df['safe_model_feature_name'].isin(feats)) & (proxy_df['near_constant_overall'])].shape[0]),
        'high_corr_pair_count': int(corr_df[(corr_df['feature_set_name'].eq(fs)) & (corr_df['high_corr_flag']) & (corr_df['feature_a'].isin(feats) | corr_df['feature_b'].isin(feats))].shape[0]),
        'vif_high_or_extreme_count': int(vif_df[(vif_df['feature_set_name'].eq(fs)) & (vif_df['safe_model_feature_name'].isin(feats)) & (vif_df['vif_risk_bucket'].isin(['high','extreme']))].shape[0]),
        'group_proxy_risk_count': int(proxy_df[(proxy_df['feature_set_name'].eq(fs)) & (proxy_df['safe_model_feature_name'].isin(feats)) & (proxy_df['group_proxy_risk'])].shape[0]),
        'leakage_suspect_count': int(leak_df[(leak_df['feature_set_name'].eq(fs)) & (leak_df['safe_model_feature_name'].isin(feats)) & (leak_df['leakage_suspect_level'].isin(['medium','high']))].shape[0]),
        'top_risk_features': joined(feats[:8]), 'summary_note': 'family-level diagnostic summary only; no feature selection decision',
    })
family_df = pd.DataFrame(family_rows)
write_csv(family_df, '10x_feature_family_distribution_summary.csv')

stage_rows = []
for (fs, stage), subcat in catalog[catalog['role'].eq('feature')].groupby(['feature_set_name','AARRR_stage'], dropna=False):
    feats = subcat['safe_model_feature_name'].tolist()
    zeros = catalog[(catalog['feature_set_name'].eq(fs)) & (catalog['safe_model_feature_name'].isin(feats))]['zero_rate']
    stage_rows.append({
        'feature_set_name': fs, 'AARRR_stage': stage, 'feature_count': len(feats),
        'numeric_count': int(subcat['inferred_feature_type'].eq('numeric').sum()), 'binary_count': int(subcat['inferred_feature_type'].eq('binary').sum()),
        'avg_zero_rate': float(zeros.mean()) if len(zeros) else np.nan,
        'near_constant_count': int(proxy_df[(proxy_df['feature_set_name'].eq(fs)) & (proxy_df['safe_model_feature_name'].isin(feats)) & (proxy_df['near_constant_overall'])].shape[0]),
        'high_corr_pair_count': int(corr_df[(corr_df['feature_set_name'].eq(fs)) & (corr_df['high_corr_flag']) & (corr_df['feature_a'].isin(feats) | corr_df['feature_b'].isin(feats))].shape[0]),
        'vif_high_or_extreme_count': int(vif_df[(vif_df['feature_set_name'].eq(fs)) & (vif_df['safe_model_feature_name'].isin(feats)) & (vif_df['vif_risk_bucket'].isin(['high','extreme']))].shape[0]),
        'group_proxy_risk_count': int(proxy_df[(proxy_df['feature_set_name'].eq(fs)) & (proxy_df['safe_model_feature_name'].isin(feats)) & (proxy_df['group_proxy_risk'])].shape[0]),
        'leakage_suspect_count': int(leak_df[(leak_df['feature_set_name'].eq(fs)) & (leak_df['safe_model_feature_name'].isin(feats)) & (leak_df['leakage_suspect_level'].isin(['medium','high']))].shape[0]),
        'interpretation_guardrail': 'Referral has no directly observed feature when applicable; diagnostics are not causal or importance claims',
        'notes': 'AARRR-stage distribution summary only',
    })
for fs in sets:
    if not ((pd.DataFrame(stage_rows)['feature_set_name'].eq(fs)) & (pd.DataFrame(stage_rows)['AARRR_stage'].eq('Referral'))).any():
        stage_rows.append({'feature_set_name': fs, 'AARRR_stage': 'Referral', 'feature_count': 0, 'numeric_count': 0, 'binary_count': 0, 'avg_zero_rate': np.nan, 'near_constant_count': 0, 'high_corr_pair_count': 0, 'vif_high_or_extreme_count': 0, 'group_proxy_risk_count': 0, 'leakage_suspect_count': 0, 'interpretation_guardrail': 'No directly observed Referral feature in current dataset; do not claim Referral was validated by data', 'notes': 'explicit empty-stage caveat'})
stage_df = pd.DataFrame(stage_rows)
write_csv(stage_df, '10x_AARRR_stage_distribution_summary.csv')
print('leakage rows:', len(leak_df), 'family/stage rows:', len(family_df), len(stage_df))

leakage rows: 101 family/stage rows: 10 9


In [11]:
key_patterns = ['watch_time_min_w3','watch_session_w3','recency','avg_gap_w3_watch_days','is_only_w1','is_cold_start_3d_fixed','is_cold_start_7d_fixed','is_user_verified','payment_is_ios','age_group','payment_is_mobile','is_female','is_premium','is_male','payment_is_pc','old_movie_ratio_5y','under_1m','under_5m','genre','ratio']
key_features = []
for c in expanded.columns:
    lc = c.lower()
    if c in key_patterns or any(p in lc for p in key_patterns):
        if c not in {'USER_KEY','is_repurchase','is_promotion'}:
            key_features.append(c)
key_features = sorted(set(key_features))
key_rows = []
for c in key_features:
    fs = 'expanded_feature_set'
    if c not in expanded.columns:
        continue
    s = to_num(expanded[c])
    dist = f"n={int(s.notna().sum())}; mean={s.mean():.4f}; q05={s.quantile(.05):.4f}; median={s.quantile(.5):.4f}; q95={s.quantile(.95):.4f}; zero_rate={(s==0).mean():.4f}" if s.notna().any() else 'non-numeric or empty'
    group_bits = []
    for gname in ['promotion_only','nonpromotion_only','promotion_repurchase','promotion_nonrepurchase','nonpromotion_repurchase','nonpromotion_nonrepurchase']:
        gs = to_num(expanded.loc[group_masks[gname], c])
        group_bits.append(f"{gname}_mean={gs.mean():.4f}" if gs.notna().any() else f"{gname}_mean=NA")
    risks = []
    if c in proxy_features: risks.append('group proxy risk review')
    if c in set(zero_tail_df[zero_tail_df['tail_risk_flag'] | zero_tail_df['zero_inflation_flag']]['safe_model_feature_name']): risks.append('zero-inflation or tail risk')
    if c in set(vif_df[vif_df['vif_risk_bucket'].isin(['high','extreme'])]['safe_model_feature_name']): risks.append('high/extreme VIF diagnostic')
    if c in set(leak_df[leak_df['leakage_suspect_level'].isin(['medium','high'])]['safe_model_feature_name']): risks.append('leakage/proxy pre-audit')
    key_rows.append({
        'safe_model_feature_name': c, 'feature_set_name': fs, 'AARRR_stage': info(fs, c, 'AARRR_stage'), 'feature_family': info(fs, c, 'feature_family'),
        'why_reviewed': '09x strong observed feature or 08x/09x proxy-risk/context/content feature family',
        'distribution_summary': dist, 'group_pattern_summary': joined(group_bits), 'risk_summary': joined(risks) or 'routine review only',
        'caveat_reason': caveat_for_feature(fs, c),
        'recommended_next_step': 'carry to 11x actual input and scope-sensitivity preflight; no selection decision in 10x',
        'not_a_feature_selection_decision': True, 'notes': 'observed distribution difference is not feature importance',
    })
key_df = pd.DataFrame(key_rows)
write_csv(key_df, '10x_key_feature_distribution_review.csv')

viz_df = pd.DataFrame([{'figure_file': '', 'figure_type': 'not_generated', 'related_feature': '', 'related_group': '', 'purpose': 'CSV diagnostics prioritized for 10x review package', 'safe_interpretation': 'No final presentation figure was created', 'unsafe_interpretation': 'Do not claim absence of figure means absence of distribution risk', 'status': 'no_figures_generated'}])
write_csv(viz_df, '10x_visualization_manifest.csv')
print('key review rows:', len(key_df))

key review rows: 38


In [12]:
caveats = [
    ('distribution_eda_only','scope','all','10x','10x is distribution EDA only.','10x proves feature importance.','Describe distributions as observed EDA patterns only.'),
    ('redundancy_pre_audit_only','scope','all','10x','Correlation/VIF/redundancy are pre-audit diagnostics.','High correlation means remove the feature.','Use removal candidate or review risk wording only.'),
    ('no_feature_removal','decision','all','10x','No feature was removed in 10x.','10x selected the final feature set.','All removal decisions require user approval.'),
    ('no_causal_claim','claim','all','10x','No causal, uplift, marketing effectiveness, model performance, or feature importance claim is made.','The feature causes churn or improves retention.','Keep claims descriptive and non-causal.'),
    ('modeling_not_performed','method','all','10x','Modeling, train/test split, prediction, SHAP, Optuna, and segmentation were not performed.','10x model results show.','Route modeling questions to 11x/12x.'),
    ('is_promotion_split_key','feature','is_promotion','10x','is_promotion is a split/scope key.','is_promotion is interpreted as an ordinary feature in 10x.','Use only for group diagnostics in this step.'),
    ('is_repurchase_target','feature','is_repurchase','10x','is_repurchase is the target label.','is_repurchase is a model feature.','Use only for 2x2 diagnostic grouping.'),
    ('is_churn_prevented_context','feature','is_churn_prevented','10x','is_churn_prevented requires target-proxy review.','is_churn_prevented is approved for modeling.','Carry to 11x preflight.'),
    ('cold_start_fixed','feature','is_cold_start_3d_fixed; is_cold_start_7d_fixed','Activation','Fixed cold-start features use row-level first-watch timing.','Original cold_start fields are model features.','Use fixed replacement caveat.'),
    ('old_movie_ratio_5y_mismatch','feature','old_movie_ratio_5y','Retention','old_movie_ratio_5y has a 9-row mismatch caveat.','old_movie_ratio_5y is fully clean.','Keep mismatch caveat in interpretation.'),
    ('genre_multi_category','feature','genre ratio features','Retention','Movie_Master same MOVIE_NUM multi-category caveat applies.','Genre ratios are exact content preference truth.','Treat as content taxonomy proxy.'),
    ('under_threshold_caveat','feature','under_1m/under_5m','Retention','under_1m/5m features are <= threshold ratios.','They prove short viewing causes churn.','Use threshold-ratio wording.'),
    ('context_proxy_risk','feature','context/profile/payment features','Acquisition/Activation','Profile/payment fields may encode structural proxy risk.','Age/payment/gender directly explains churn.','Carry to 11x sensitivity and SHAP guardrail.'),
]
caveat_df = pd.DataFrame(caveats, columns=['caveat_id','caveat_topic','applies_to_feature','applies_to_stage','safe_claim','unsafe_claim','required_wording'])
write_csv(caveat_df, '10x_caveat_and_claim_guardrail.csv')

handoff_rows = [
    ('11x modeling preflight','conservative vs expanded feature usage','both','all model features','Need explicit scope decision before modeling','Save and inspect actual model input feature list','Expanded features may be used unintentionally',True,'Do not proceed directly to Optuna/SHAP/segmentation'),
    ('11x modeling preflight','actual model input feature list','expanded_feature_set','expanded 80 feature expectation','Verify expanded features actually used','Persist input list per scope','Model evidence may not match claimed feature set',True,''),
    ('11x modeling preflight','near-constant/group-proxy sensitivity','expanded_feature_set','is_user_verified; payment_is_ios; age_group; payment_is_mobile; is_female; is_premium; is_male; payment_is_pc','Proxy-risk features need scope review','Run approved sensitivity checks only in modeling step','Structural proxy could dominate interpretation',True,''),
    ('11x modeling preflight','LogisticRegression VIF/redundancy risk','both','high VIF and high correlation clusters','Coefficient interpretation may be unstable','Carry VIF cluster table into preflight','Coefficient sign/size may be overread',True,''),
    ('12x model comparison','model family redundancy sensitivity','both','correlated clusters','Different model families react differently to redundancy','Compare family-level behavior cautiously','AUC-only interpretation may hide redundancy effects',True,''),
    ('SHAP','correlated feature importance split','both','redundancy clusters','SHAP attribution may split among correlated features','Interpret as feature families/clusters','Single-feature importance overclaim',True,''),
    ('segmentation','do not name segment from feature name first','both','all high-risk features','Segments require rule and distribution validation','Use provisional labels after distribution/rule check','Unsafe customer labeling',True,''),
]
handoff_df = pd.DataFrame(handoff_rows, columns=['downstream_step','handoff_topic','feature_set_name','related_features','reason','required_action','risk_if_ignored','user_approval_required','notes'])
write_csv(handoff_df, '10x_downstream_handoff.csv')

risk_rows = []
risk_id = 1
def add_risk(topic, feats, fs, rtype, severity, evidence, req):
    global risk_id
    risk_rows.append({'risk_id': f'10x_R{risk_id:03d}', 'risk_topic': topic, 'related_features': feats, 'feature_set_name': fs, 'risk_type': rtype, 'severity': severity, 'evidence': evidence, 'required_check_before_modeling': req, 'allowed_action_now': 'record as review risk only', 'forbidden_action_now': 'feature removal, feature selection decision, causal/importance claim', 'user_approval_required': True, 'notes': '10x pre-audit risk register'})
    risk_id += 1
for _, r in proxy_df[proxy_df['near_constant_overall'] | proxy_df['near_constant_any_group']].head(30).iterrows(): add_risk('near-constant diagnostic', r['safe_model_feature_name'], r['feature_set_name'], 'near_constant', 'medium', 'near_constant threshold met overall or in group', 'Check whether feature remains useful and allowed per scope')
for _, r in proxy_df[proxy_df['group_proxy_risk']].head(30).iterrows(): add_risk('group proxy diagnostic', r['safe_model_feature_name'], r['feature_set_name'], 'group_proxy', 'high', 'context/profile/payment or group spread risk', 'Run 11x sensitivity and interpretation guardrail')
for _, r in corr_df[corr_df['high_corr_flag']].head(30).iterrows(): add_risk('high correlation pair', f"{r['feature_a']}; {r['feature_b']}", r['feature_set_name'], 'high_correlation', 'medium', 'abs pearson or spearman >= 0.85', 'Review cluster/family before interpreting coefficients or SHAP')
for _, r in vif_df[vif_df['vif_risk_bucket'].isin(['high','extreme'])].head(30).iterrows(): add_risk('high VIF diagnostic', r['safe_model_feature_name'], r['feature_set_name'], 'high_vif', 'medium', f"VIF bucket {r['vif_risk_bucket']}", 'Consider model-family-specific interpretation caution')
for _, r in dup_df[dup_df['duplicate_like_risk']].head(30).iterrows(): add_risk('duplicate-like feature diagnostic', f"{r['feature_a']}; {r['feature_b']}", r['feature_set_name'], 'duplicate_like', 'medium', r['reason'], 'Review semantic duplication before modeling')
for _, r in leak_df[leak_df['leakage_suspect_level'].isin(['medium','high'])].head(30).iterrows(): add_risk('leakage or target proxy pre-audit', r['safe_model_feature_name'], r['feature_set_name'], 'leakage_suspect' if r['leakage_suspect_level'] == 'high' else 'target_proxy', 'high' if r['leakage_suspect_level'] == 'high' else 'medium', r['reason_for_review'], 'Verify day21 availability and target proxy policy before modeling')
for _, r in cluster_df.head(30).iterrows(): add_risk('SHAP correlated-family interpretation risk', r['cluster_feature_list'], r['feature_set_name'], 'SHAP_interpretation_risk', 'medium', r['reason'], 'Interpret SHAP by family/cluster, not isolated feature only')
add_risk('scope policy risk', 'conservative_safe_22; expanded_feature_set', 'both', 'scope_policy_risk', 'high', 'Feature use must follow approved 06x/07x scope policy', 'Save actual model input list and request user approval for scope-sensitive decisions')
risk_df = pd.DataFrame(risk_rows)
write_csv(risk_df, '10x_modeling_preflight_risk_register.csv')
print('guardrails/handoff/risk rows:', len(caveat_df), len(handoff_df), len(risk_df))

guardrails/handoff/risk rows: 13 7 118


In [13]:
ended_at = datetime.now().isoformat(timespec='seconds')
fp_rows = []
for role, path in raw_files:
    before = fp_before[str(path)]
    after = file_fingerprint(path)
    if not before['exists']:
        status = 'missing_before'
    elif not after['exists']:
        status = 'missing_after'
    elif before.get('error') or after.get('error'):
        status = 'error'
    elif before['sha256'] == after['sha256'] and before['mtime'] == after['mtime'] and before['size'] == after['size']:
        status = 'unchanged'
    else:
        status = 'changed'
    fp_rows.append({'file_path': str(path), 'file_role': role, 'sha256_before': before['sha256'], 'sha256_after': after['sha256'], 'mtime_before': before['mtime'], 'mtime_after': after['mtime'], 'size_before': before['size'], 'size_after': after['size'], 'status': status})
fp_df = pd.DataFrame(fp_rows)
write_csv(fp_df, '10x_source_fingerprint_before_after.csv')

readme = f'''# 10x feature distribution redundancy pre-audit

## Purpose
10x is a feature distribution EDA plus redundancy / group-proxy pre-audit step after 09x. It uses the 06x conservative and expanded datasets, 07x AARRR mapping, 08x promotion vs nonpromotion EDA, and 09x promotion x repurchase 2x2 EDA outputs.

## What 10x does
- Profiles conservative_safe_22 and expanded_feature_set distributions overall, by promotion split, and by promotion x repurchase 2x2 cohorts.
- Audits zero-inflation, heavy-tail, outlier, sparse binary, near-constant, pairwise correlation, VIF, duplicate-like risk, group-proxy risk, and leakage-suspect candidates.
- Carries content/genre caveats, old_movie_ratio_5y 9-row mismatch caveat, cold_start_fixed caveat, and context/profile/payment proxy-risk caveat forward.
- Creates handoff tables for 11x / 12x / SHAP / segmentation review.

## What 10x does not do
10x does not perform modeling, target prediction, train/test split, SHAP, Optuna, segmentation, final segment creation, final business recommendation, causal claim, feature importance claim, feature removal, or feature selection decision.

## Strengthened relative to master plan 7-10
This step adds a stricter source fingerprint before/after check, 2x2 distribution review, VIF diagnostic, redundancy cluster pre-audit, duplicate-like feature audit, group-proxy review, target-leakage suspect pre-audit, and a modeling preflight risk register.

## Conservative / expanded distinction
- conservative_safe_22 rows: {conservative.shape[0]}, columns: {conservative.shape[1]}
- expanded_feature_set rows: {expanded.shape[0]}, columns: {expanded.shape[1]}
- Row alignment: {alignment}

## Diagnostic summary
- Numeric overall rows: {len(num_overall_df)}
- Binary overall rows: {len(bin_overall_df)}
- Group numeric rows: {len(num_group_df)}
- Group binary rows: {len(bin_group_df)}
- Zero/tail risk rows: {len(zero_tail_df)}
- Near-constant/group-proxy rows: {len(proxy_df)}
- Pairwise correlation rows: {len(corr_df)}
- Redundancy clusters: {len(cluster_df)}
- VIF rows: {len(vif_df)}
- Duplicate-like rows: {len(dup_df)}
- Leakage suspect rows: {len(leak_df)}

## Interpretation caveats
Distribution differences are not feature importance. Correlation and VIF do not imply automatic removal. Group-proxy risk does not imply automatic exclusion. Referral has no directly observed feature and must not be claimed as data-validated.

## Downstream handoff
The next step is 11x modeling preflight or 11x baseline growth comparison, not direct Optuna, SHAP, or segmentation. 11x must save and inspect the actual model input feature list, verify whether expanded features were actually used, review near-constant/group-proxy sensitivity, and carry redundancy clusters into interpretation guardrails. SHAP should interpret correlated features by family/cluster. Segmentation should validate rule and distribution before provisional naming.
'''
(OUT / 'README.md').write_text(readme, encoding='utf-8')

note_append = f'''

## 10x 수행 기록: feature distribution redundancy pre-audit 260516

- 10x를 수행했다. 이번 단계는 feature distribution EDA + redundancy / group-proxy pre-audit 단계였다.
- 모델링, SHAP, Optuna, segmentation은 수행하지 않았다.
- 06x conservative / expanded dataset을 입력으로 사용했다.
- 07x AARRR mapping을 입력으로 사용했다.
- 08x promotion vs nonpromotion EDA 결과를 입력으로 사용했다.
- 09x promotion x repurchase 2x2 EDA 결과를 입력으로 사용했다.
- feature distribution, zero-inflation, outlier, near-constant, correlation, VIF, redundancy, group-proxy risk를 진단했다.
- feature 제거 결정이 아니다.
- feature selection 결정이 아니다.
- 사용자 승인 없이 feature를 제거하지 않는다.
- 11x modeling preflight에서 actual model input feature list를 반드시 검수해야 한다.
- SHAP 단계에서는 correlated feature를 family 단위로 해석해야 한다.
- segmentation 단계에서는 이름보다 기준식과 분포 확인이 먼저다.
- 다음 단계는 11x modeling preflight / baseline growth comparison이다.
'''
note_path = PARK / 'note.md'
note_text = note_path.read_text(encoding='utf-8') if note_path.exists() else ''
if '## 10x 수행 기록: feature distribution redundancy pre-audit 260516' not in note_text:
    note_path.write_text(note_text.rstrip() + note_append, encoding='utf-8')
(OUT / 'note_tail_copy.md').write_text('\n'.join((PARK / 'note.md').read_text(encoding='utf-8').splitlines()[-80:]), encoding='utf-8')

log_lines = [
    f'execution_start={started_at}', f'execution_end={ended_at}', f'notebook_path={NOTEBOOK_PATH}', f'executed_notebook_path={NOTEBOOK_PATH}',
    'input_file_load_status=loaded 06x/07x/08x/09x required inputs', 'output_file_creation_status=created CSV/README/note copy outputs',
    'visible_output_save_status=checked from on-disk notebook before final zip when available',
    'warnings=' + joined(warnings_list), 'errors=' + joined(errors_list), 'final_status=pending final check table below',
]
(OUT / '10x_execution_log.txt').write_text('\n'.join(log_lines), encoding='utf-8')
print('documentation and fingerprint written:', ended_at)

documentation and fingerprint written: 2026-05-16T02:36:50


In [14]:
def notebook_has_visible_outputs(path):
    if not path.exists():
        return False
    try:
        nb = json.loads(path.read_text(encoding='utf-8'))
        code_cells = [c for c in nb.get('cells', []) if c.get('cell_type') == 'code']
        has_exec = any(c.get('execution_count') is not None for c in code_cells)
        has_output = any(len(c.get('outputs', [])) > 0 for c in code_cells)
        return bool(has_exec and has_output)
    except Exception as exc:
        warnings_list.append(f'notebook output inspection failed: {exc}')
        return False

output_csvs = sorted(OUT.glob('10x_*.csv'))
required_items = []
required_items.append(('notebook', NOTEBOOK_PATH, f'notebook/{STEP}/{STEP}.ipynb'))
required_items.append(('executed notebook with visible outputs', NOTEBOOK_PATH, f'notebook/{STEP}/{STEP}.ipynb'))
for p in output_csvs:
    required_items.append((p.name, p, f'reports/audits/{STEP}/{p.name}'))
for p in [OUT / 'README.md', OUT / '10x_execution_log.txt', OUT / 'note_tail_copy.md']:
    required_items.append((p.name, p, f'reports/audits/{STEP}/{p.name}'))
for p in FIG_DIR.glob('*'):
    if p.is_file():
        required_items.append(('figure', p, f'reports/audits/{STEP}/figures/{p.name}'))

inventory_rows = []
for item, p, arc in required_items:
    exists = p.exists()
    inventory_rows.append({'required_item': item, 'expected_path_in_zip': arc, 'exists': exists, 'size_bytes': p.stat().st_size if exists else 0, 'status': 'PASS' if exists and (item != 'executed notebook with visible outputs' or notebook_has_visible_outputs(p)) else 'FAIL'})
inv_df = pd.DataFrame(inventory_rows)
write_csv(inv_df, '10x_review_zip_inventory.csv')

if ZIP_PATH.exists():
    ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for _, p, arc in required_items:
        if p.exists():
            zf.write(p, arc)
    inv_path = OUT / '10x_review_zip_inventory.csv'
    if inv_path.exists():
        zf.write(inv_path, f'reports/audits/{STEP}/10x_review_zip_inventory.csv')

output_names = {p.name for p in OUT.glob('*') if p.is_file()}
checks = []
def add_check(name, ok, detail=''):
    checks.append({'check': name, 'status': 'PASS' if ok else 'FAIL', 'detail': detail})

raw_unchanged = fp_df['status'].eq('unchanged').all()
inside = str(OUT).startswith(str(PARK)) and str(NOTEBOOK_PATH).startswith(str(PARK)) and str(ZIP_PATH).startswith(str(PARK))
add_check('all_outputs_inside_park_ingyeom', inside, str(PARK))
add_check('raw_source_csv_not_modified_by_sha256', raw_unchanged, joined(fp_df.loc[~fp_df['status'].eq('unchanged'), 'file_path'].tolist()))
add_check('source_fingerprint_created', '10x_source_fingerprint_before_after.csv' in output_names)
add_check('notebook_exists', NOTEBOOK_PATH.exists(), str(NOTEBOOK_PATH))
add_check('notebook_executed', notebook_has_visible_outputs(NOTEBOOK_PATH), 'execution_count and outputs present in on-disk notebook')
add_check('executed_notebook_visible_outputs_saved', notebook_has_visible_outputs(NOTEBOOK_PATH), 'same path included as executed notebook')
add_check('execution_log_created', (OUT / '10x_execution_log.txt').exists())
for n, folder in [('06x_inputs_loaded', F06), ('07x_inputs_loaded', F07), ('08x_inputs_loaded', F08), ('09x_inputs_loaded', F09)]: add_check(n, folder.exists())
for n, p in [('06x_final_checks_pass', F06/'06x_final_checks.csv'), ('07x_final_checks_pass', F07/'07x_final_checks.csv'), ('08x_final_checks_pass', F08/'08x_final_checks.csv'), ('09x_final_checks_pass', F09/'09x_final_checks.csv')]: add_check(n, final_checks_pass(p))
add_check('conservative_dataset_loaded', not conservative.empty, str(conservative.shape))
add_check('expanded_dataset_loaded', not expanded.empty, str(expanded.shape))
add_check('conservative_expanded_row_alignment_verified', alignment)
for chk, fname in [
    ('feature_distribution_catalog_created','10x_feature_distribution_catalog.csv'), ('numeric_distribution_overall_created','10x_numeric_distribution_overall.csv'), ('binary_distribution_overall_created','10x_binary_distribution_overall.csv'),
    ('numeric_distribution_by_group_created','10x_numeric_distribution_by_group.csv'), ('binary_distribution_by_group_created','10x_binary_distribution_by_group.csv'), ('zero_inflation_tail_risk_audit_created','10x_zero_inflation_and_tail_risk_audit.csv'),
    ('near_constant_group_proxy_audit_created','10x_near_constant_and_group_proxy_audit.csv'), ('pairwise_correlation_audit_created','10x_pairwise_correlation_audit.csv'), ('redundancy_cluster_pre_audit_created','10x_redundancy_cluster_pre_audit.csv'),
    ('vif_pre_audit_created','10x_vif_pre_audit.csv'), ('duplicate_like_feature_audit_created','10x_duplicate_like_feature_audit.csv'), ('target_leakage_suspect_pre_audit_created','10x_target_leakage_suspect_pre_audit.csv'),
    ('feature_family_distribution_summary_created','10x_feature_family_distribution_summary.csv'), ('AARRR_stage_distribution_summary_created','10x_AARRR_stage_distribution_summary.csv'), ('key_feature_distribution_review_created','10x_key_feature_distribution_review.csv'),
    ('caveat_claim_guardrail_created','10x_caveat_and_claim_guardrail.csv'), ('downstream_handoff_created','10x_downstream_handoff.csv'), ('modeling_preflight_risk_register_created','10x_modeling_preflight_risk_register.csv')]: add_check(chk, fname in output_names)
for n in ['no_modeling_performed','no_train_test_split_performed','no_prediction_performed','no_shap_performed','no_optuna_performed','no_segmentation_performed','no_final_business_claim_created','no_new_features_created','no_feature_removed','no_feature_selection_decision_made']:
    add_check(n, True, '10x notebook only generated diagnostics and audit tables')
add_check('README_created', (OUT / 'README.md').exists())
add_check('note_md_updated', '10x 수행 기록: feature distribution redundancy pre-audit 260516' in (PARK / 'note.md').read_text(encoding='utf-8'))
add_check('review_zip_inventory_created', (OUT / '10x_review_zip_inventory.csv').exists())
add_check('review_zip_created', ZIP_PATH.exists(), str(ZIP_PATH))
fail_count = sum(1 for c in checks if c['status'] != 'PASS')
add_check('critical_fail_count_zero', fail_count == 0, str(fail_count))
final_df = pd.DataFrame(checks)
write_csv(final_df, '10x_final_checks.csv')

print('review zip:', ZIP_PATH, ZIP_PATH.exists())
print('final check status counts:')
print(final_df['status'].value_counts().to_string())
display(final_df.tail(8))

review zip:

 C:\Code\ott-churn-prediction\park.ingyeom\zip\10x_feature_distribution_redundancy_pre_audit_260516_review_package.zip True
final check status counts:
status
PASS    51


C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\zipfile.py:1566: UserWarning: Duplicate name: 'notebook/10x_feature_distribution_redundancy_pre_audit_260516/10x_feature_distribution_redundancy_pre_audit_260516.ipynb'
  return self._open_to_write(zinfo, force_zip64=force_zip64)
C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\zipfile.py:1566: UserWarning: Duplicate name: 'reports/audits/10x_feature_distribution_redundancy_pre_audit_260516/10x_review_zip_inventory.csv'
  return self._open_to_write(zinfo, force_zip64=force_zip64)


,check,status,detail
43,no_new_features_created,PASS,10x notebook only generated diagnostics and au...
44,no_feature_removed,PASS,10x notebook only generated diagnostics and au...
45,no_feature_selection_decision_made,PASS,10x notebook only generated diagnostics and au...
46,README_created,PASS,
47,note_md_updated,PASS,
48,review_zip_inventory_created,PASS,
49,review_zip_created,PASS,C:\Code\ott-churn-prediction\park.ingyeom\zip\...
50,critical_fail_count_zero,PASS,0
